# Welcome to Foundations of TEM Data Management and Analysis 2026: A Hands-on CITEAM Workshop One-day program 

The 2026 CITEAM workshop is geared toward cyberinfrastructure (CI) training for the community of scientists and researchers using the new generation of Transmission Electron Microscope (TEM) instruments at UMD. The training materials presented during the workshop and the associated training activities will help reduce barriers to the use of the instruments, data, software tools, and computing to help users in key steps critical to their scientific inquiry - from data collection, data analysis, to storing and publishing data - encompassing end-to-end TEM Data Management.

This edition of the workshop includes a keynote talk from Professor Ichiro Takeuchi, and cover foundational materials relevant to the instrument users.

The day-long workshop will be structured around a mix of lectures introducing the materials customized for the instrument users, hands-on sessions where participants will have the opportunity to go through end-to-end examples with the workshop instructors, and a site visit to the actual instrument. The instructors will be available this afternoon at "office hours" to address specific questions and concerns.

## Section 0.1.1 - First Steps

In this introductory section, we will open a single TIFF micrograph and a stack of TIFF frames, inspect basic image properties, and generate a quick JPEG thumbnail for browsing and sharing. This will be the first step in virtually all of your microscopy workflows.




In [ ]:
# First, import the modules we'll be using later. All of this Python code was written
# by a multitude of other people and will do an enormous amount of work "behind the scenes".

# These are some of the standard packages used by most Python programs
import os, re, json, hashlib
from datetime import datetime
from pathlib import Path

# The next packages are specialized ones for math, drawing, and handling image files.
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
import h5py   # Used for HDF5, if we decide to actually use that.
from scipy import ndimage
from scipy.signal import fftconvolve

#Finally, Hyperspy is a package that provides algorithms specifically for microscopy
import hyperspy.api as hs

# Now, let's set some variables to the filenames where our images are stored.
# We'll be using these a LOT today, so defining them this way does two things. First, it
# saves us a lot of typing. Secondly, if we move our files or want to change which ones we're
# working with, we have to make only one change here and everything below here
# will still work as normal.

#Alex:
#SINGLE_TIFF_PATH = "/Users/alexh/Desktop/20251024/JEM-ARM200FImage_20251024_1518_44_ADF1_ImagePanel1.tif"          # e.g., "data/single/micrograph_01.tif"
#STACK_FOLDER_PATH = "/Users/alexh/Desktop/20251024/JEM-ARM200FImage_20251024_1518_51_ADF1_ImagePanel1"         # e.g., "data/stack_01_frames/"
#OUTPUT_DIR = "outputs_0_2"
#Erik
SINGLE_TIFF_PATH = "/Users/escott/projects/working-CITEAM/SiGe/JEM-ARM200FImage_20251210_1036_43_ADF1_ImagePanel1/JEM-ARM200FImage_20251210_1036_46_ADF1_1_ImagePanel1.tif"          # e.g., "data/single/micrograph_01.tif"
STACK_FOLDER_PATH = "/Users/escott/projects/working-CITEAM/SiGe/JEM-ARM200FImage_20251210_1036_43_ADF1_ImagePanel1"         # e.g., "data/stack_01_frames/"
OUTPUT_DIR = "outputs_0_1"
# Others can go here. We'll reduce this to one set of values once we know the exact configuration of
# the lab desktops.

# Create the output directory. If it already exists, that's fine, just go on. If need be,
# create any parent directories.
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


# Let's load an image WITHOUT using Hyperspy to get an appreciation for just how much work Hyperspy does for us.
# We'll define a function called "image_stats()" that will collect some basic information about the image and
# return it in JSON format. Then we'll actually load the image, call that function, and print out the results.

def image_stats(img2d):
    img = img2d.astype(float)
    return {
        "shape": list(img.shape),
        "dtype": str(img2d.dtype),
        "min": float(np.min(img)),
        "max": float(np.max(img)),
        "mean": float(np.mean(img)),
        "std": float(np.std(img)),
        "p1": float(np.percentile(img, 1)),
        "p50": float(np.percentile(img, 50)),
        "p99": float(np.percentile(img, 99))
    }


img = tiff.imread(SINGLE_TIFF_PATH)
img = img.astype(float)
print("Single image stats:", json.dumps(image_stats(img), indent=2))


# To see the results of our handiwork so far, let's plot that image into our notebook here.
plt.figure(figsize=(5,5)); plt.imshow(img); plt.title("0.1.1 Single TIFF"); plt.axis("off"); plt.show()

# And finally, for this section, create a "thumbnail" image and save it as a jpeg (instead of a TIFF file).
# This will create a function called "save_jpeg_thumbnail()" to do the actual work.

def save_jpeg_thumbnail(img2d, out_path, p_low=1, p_high=99, max_size=512):
    img = img2d.astype(float)
    lo, hi = np.percentile(img, [p_low, p_high])
    img = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)
    H, W = img.shape
    scale = max(H, W) / max_size
    if scale > 1:
        step = int(np.ceil(scale))
        img = img[::step, ::step]
    plt.imsave(out_path, img, cmap=None)
thumb_path = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + "_thumb.jpg")
save_jpeg_thumbnail(img, thumb_path)
print("Saved thumbnail:", thumb_path)


# Now, let's do this again using Hyperspy. Hyperspy knows how to deal with multiple input file types,
# including HDF5 and Digital Micrograph, and can read literally dozens of image file formats.

print("=========")
print("Loading an image using Hyperspy")
s = hs.load(SINGLE_TIFF_PATH)
print("loaded successfully!")


## Section 0.1.2 - Histogram and Contrast

In this section, we will plot histograms, inspect intensity distributions, and apply contrast adjustment and optional gamma correction. Here we are looking for areas of interest and/or concern in the collected images - both correcting for non-linear response in both the instruments and in human vision and also detecting and adjusting pixels that are down in the noise or full-scale high.

Let's use the hyperspy function "plot_histograms()" to produce and draw them. We'll experiment with a few different parameters.



In [ ]:
# First, we'll just take the defaults
hs.plot.plot_histograms(s[0])

# Let's try some different options for binning.

# We could try the square root method (the one Excel uses, BTW)
hs.plot.plot_histograms(s[0], legend=["sqr root binning"], bins="sqrt")

# Or Knuth's algorithm...
hs.plot.plot_histograms(s[0], legend=["Knuth's binning"], bins="knuth")


# There are many more binning algorithms available - see https://hyperspy.org/hyperspy-doc/current/reference/api.plot/index.html

# Let's plot the image again, this time using Hyperspy's built-in plotting instead of relying on the underlying
# matplotlib routines by themselves.

s[0].plot()
print(type(s[0].data[0][0]))


p_low, p_high = 1, 99
lo, hi = np.percentile(s[0].data, [p_low, p_high])
print("lo =", lo, " hi =", hi)
img_cs = np.clip((s[0].data - lo) / (hi - lo + 1e-12), 0, 1)
print(type(s[0].data[0][0]))
s[0].data = img_cs
print(type(s[0].data[0][0]))

hs.plot.plot_histograms(s[0], legend=["auto binning, clipped"], bins="auto")

# Finally, let's adjust the contrast of the image by changing the image's "gamma". This is a mapping from the
# linearity of the detector to the non-linearities of the screen we are looking at combined with the
# non-linearities of human vision. 

gamma = 0.7  # try 0.25, 0.5, 1.0, 1.5, and 2
img_out = np.clip(img_cs ** gamma, 0, 1)
s[0].data = img_out
print(type(s[0].data[0][0]))

hs.plot.plot_histograms(s[0], legend=["auto, clipped, gamma"], bins="auto")
s[0].plot()

## Section 0.1.3 - Exercise 0.1.3: Noise correction (Gaussian blur)

In this section, we will use some algorithms to remove noise from images (including Gaussian denoising), compare before/after images, and discuss the tradeoff between noise reduction and feature preservation.




In [ ]:
# We've messed with the data in "s", specifically s[0], quite a lot. Depending on what we
# might have done during the break, there's a good chance it's a mess. So let's re-load it!

s = hs.load(SINGLE_TIFF_PATH)

import copy
img = copy.deepcopy(s[0].data)

# We'll clip as before to handle the extreme ends of the range
lo, hi = np.percentile(img, [1, 99])
vis = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)

# Now we'll use SciPy's "ndimage" package to apply a Gaussian filter to the underlying data

sigma = 0.5  # try 0.5, 1.0, 2.0
dn = ndimage.gaussian_filter(vis, sigma=sigma)

plt.figure(figsize=(5,5)); plt.imshow(vis); plt.title("0.1.3 Before"); plt.axis("off"); plt.show()
plt.figure(figsize=(5,5)); plt.imshow(dn); plt.title(f"0.1.3 After Gaussian (sigma={sigma})"); plt.axis("off"); plt.show()

# Let's try successively larger values of sigma - notice the list syntax for for loops
for sigma in [0.25, 0.5, 1.0, 2.0]:
    dn = ndimage.gaussian_filter(vis, sigma=sigma)
    plt.figure(figsize=(5,5)); plt.imshow(dn); plt.title(f"0.1.3 After Gaussian (sigma={sigma})"); plt.axis("off"); plt.show()


# We can implement the noise filtering algorithms of our choice, or we can
# let Hyperspy do the hard work for us...

# start with a clean copy again:

s[0].change_dtype('float')

s[0].decomposition()    # taking ALL the defaults - a lot to understand here.

s[0].plot()


# Section 0.1.4 - Exercise 0.1.4: Drift correction on a stack (cross-correlation registration)

In this section we will identify drift in sequential TEM frames, register a stack, and compare corrected and uncorrected results. This is necessary because as the microscope takes successive frames, there is no chance in the world the images will be perfectly aligned with each other.

"Any sufficiently precise instrument **IS** a thermometer" - some scientist or engineer, once upon a time, somewhere.


In [ ]:
# The first requirement of drift correction is to have more than one image.
# So far we've just been working with the one, but we have a directory with ten of them.
# Let's use them. It's mildly tedious to do this by hand, but again Hyperspy makes life easy.

tiffFilesWildcard = STACK_FOLDER_PATH + "/*.tif"
print("loading all the files matching", tiffFilesWildcard)
imgStack = hs.load(tiffFilesWildcard, stack=True)

#print(imgStack[0])
#imgStack[0]

print('The type of "imgStack" is', type(imgStack))
print('The length of that list (aka "array") is', len(imgStack))
print('A compact representation of "imgStack" is', imgStack[0])
print("=================")
print("The basic metadata for the first image is:")
print("=================")
print(imgStack[0].metadata)

# Now let's do our drift correction (image stacking)
# all in one line of code!

imgStack[0].align2D()
imgStack[0].plot()


## Section 0.1.5 - Measure distances (pixels → nm) using the scale bar

In this section we will measure a feature in pixels, calibrate using the image scale bar, and convert the result into nanometers. There are, as always, several ways to do this. We could measure features in terms of pixels and then convert them to nanometers, or we could tell Hyperspy what the image scale is and have it give us a measurement tool that reads directly in our chosen units. We'll do the latter here.


In [ ]:
# First, we need to open the Hyperspy "axes manager" and enter the values for the scale.
# Run this code cell, and three clickable buttons will appear below: "Axis 1", "Axis 2", and "Axis 3".
# Axis 1 is just the number of images in the stack - 10. Axis 2 is the width axis and axis 3 is the height axis.
# Click on each of those two, enter "nm" for your units, and .0390625000 for the scale. In other words,
# each pixel represents .039-ish nanometers, or 25.6 pixels per nanometer

imgStack[0].axes_manager.gui()

In [ ]:
# Now we can plot the data. Take a look at the scale mark in the bottom left corner of the image.

imgStack[0].plot()

In [ ]:
# We don't have to hold a ruler up to the screen. Hyperspy provides a handy "calibrate()" function that lets us
# draw a line and it will report the length in our chose scale units. (If we hadn't set the axes scales above then this
# would just return the number of pixels and we would have to do something sensible with it.
imgStack[0].plot()
imgStack[0].calibrate(interactive=True)